# 01_Normalize — Unification des sources brutes

**Ce qui a changé par rapport à la version précédente** : l'ancienne version
de ce notebook lisait des champs qui n'existent pas dans les fichiers réels
(`item.get('service_ar')`, `org_data['services'].items()`...) et échouait
silencieusement (ou plantait) sans jamais fusionner les 58 fiches des 4
fichiers population avec les 4 tableaux arabes officiels. Ce notebook a été
réécrit après inspection directe de la structure réelle de chaque fichier de
`rag data/`.

**Stratégie de matching intelligent pour les services** (double source) :
1. Les **4 tableaux arabes** (`جدول خدمات ...json`) sont la source la plus
   riche et la plus officielle (colonnes explicites : population, axe,
   service, institutions, description, conditions, documents) — priorité
   absolue sur les libellés, conformément au guide.
2. Les **4 fichiers population** (`Services_enfants.json`, etc., structure
   `{"fiche": {...}}`) sont fusionnés avec les tableaux par correspondance
   floue (`rapidfuzz`) sur le nom de service normalisé, au sein de la même
   population cible. En cas de match : le libellé du tableau arabe est
   conservé, mais conditions/documents sont **unionés** (pas remplacés) —
   les fiches contiennent souvent des listes de documents plus complètes.
   Les entrées non matchées de chaque source sont conservées telles quelles.
3. `Services_Sociaux_Organisé.json` est un **manifeste** (`services`: liste
   de 4 noms de catégories, pas de données) — utilisé uniquement pour
   valider le compte final (`metadata.total_services`), jamais comme source
   de contenu.

In [1]:
import sys, json
from pathlib import Path
from collections import defaultdict
from rapidfuzz import fuzz

sys.path.insert(0, str(Path.cwd()))
from etl_lib.ontology import (
    INSTITUTION_MAP, POPULATION_CIBLES, MOROCCAN_REGIONS, EPS_CODE,
    clean_text, normalize_arabic, stable_id, extract_institutions, normalize_region,
)
from etl_lib.io_utils import load_json, stream_jsonl_repair, save_jsonl

RAG_DATA_DIR = Path.cwd().parent / "rag data"
PROCESSED_DIR = Path.cwd().parent / "data" / "processed"
REPORTS_DIR = Path.cwd().parent / "data" / "reports"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)


def get_field(d: dict, *substrings: str) -> str:
    """Cherche une clé du dict contenant l'une des sous-chaînes données
    (tolère les variations d'espaces/orthographe entre les 4 fichiers
    tableaux arabes, créés à la main indépendamment)."""
    for key, value in d.items():
        key_clean = clean_text(key)
        if any(sub in key_clean for sub in substrings):
            return value
    return ""


def split_lines(text) -> list:
    if isinstance(text, list):
        return [clean_text(t) for t in text if clean_text(t)]
    text = clean_text(text)
    if not text:
        return []
    parts = [p.strip(" -\u2022") for p in text.replace("\n", "|").split("|")]
    return [p for p in parts if p]


## 1a. Services — Source 1 : les 4 tableaux arabes officiels

In [2]:
ARABIC_TABLES = {
    "enfants": "جدول خدمات الأطفال-Enfants.json",
    "femmes": "جدول خدمات - النساء Femmes .json",
    "personnes_agees": "جدول خدمات -PA المسنين.json",
    "personnes_handicapees": "جدول خدمات  PSH.json",
}


def parse_arabic_tables() -> list:
    records = []
    for population, fname in ARABIC_TABLES.items():
        rows = load_json(RAG_DATA_DIR / fname)
        if not rows:
            print(f"  \u26a0 {fname} introuvable ou invalide")
            continue
        n_before = len(records)
        for row in rows:
            service_ar = clean_text(get_field(row, "البرامج", "الخدمات", "الخدمة"))
            if not service_ar:
                continue
            institutions_raw = get_field(row, "المؤسسات", "المراكز")
            institutions, eps_fallback = extract_institutions(institutions_raw)
            records.append({
                "source": "table_arabe",
                "population_cible": population,
                "axe_programme_ar": clean_text(get_field(row, "المحور", "البرنامج العام")),
                "service_ar": service_ar,
                "institutions_raw": institutions_raw,
                "institutions": institutions,
                "eps_fallback": eps_fallback,
                "description_ar": clean_text(get_field(row, "وصف الخدمة", "الوصف")),
                "conditions_ar": split_lines(get_field(row, "شروط", "الاستفادة")),
                "documents_ar": split_lines(get_field(row, "الوثائق", "المطلوبة")),
            })
        print(f"  \u2713 {fname}: {len(records) - n_before} services")
    return records


table_services = parse_arabic_tables()
print(f"\nTotal tableaux arabes : {len(table_services)} services")


  ✓ جدول خدمات الأطفال-Enfants.json: 13 services
  ✓ جدول خدمات - النساء Femmes .json: 13 services
  ✓ جدول خدمات -PA المسنين.json: 11 services
  ✓ جدول خدمات  PSH.json: 22 services

Total tableaux arabes : 59 services


## 1b. Services — Source 2 : les 4 fichiers population (structure `fiche`)

In [3]:
POPULATION_FILES = {
    "enfants": "Services_enfants.json",
    "femmes": "Services_femmes.json",
    "personnes_agees": "Services_personnes_agees.json",
    "personnes_handicapees": "Services_personnes_handicapees.json",
}


def parse_population_fiches() -> list:
    records = []
    for population, fname in POPULATION_FILES.items():
        data = load_json(RAG_DATA_DIR / fname)
        if not data or "services" not in data:
            print(f"  \u26a0 {fname} introuvable ou invalide")
            continue
        items = data["services"]
        n_before = len(records)
        for item in items:
            fiche = item.get("fiche", {})
            # 2 variantes de schema selon le fichier : Services_personnes_handicapees.json
            # utilise service/centre_ou_programme, les 3 autres utilisent
            # programme_service/institutions -- on tolère les deux.
            service_ar = clean_text(fiche.get("service") or fiche.get("programme_service", ""))
            if not service_ar:
                continue
            institutions_raw = fiche.get("centre_ou_programme") or fiche.get("institutions", "")
            institutions, eps_fallback = extract_institutions(institutions_raw)
            records.append({
                "source": "fiche_population",
                "raw_id": item.get("id", ""),
                "population_cible": item.get("categorie", population),
                "service_ar": service_ar,
                "institutions_raw": institutions_raw,
                "institutions": institutions,
                "eps_fallback": eps_fallback,
                "description_ar": clean_text(fiche.get("description", "")),
                "conditions_ar": split_lines(fiche.get("conditions", [])),
                "documents_ar": split_lines(fiche.get("documents", [])),
            })
        print(f"  \u2713 {fname}: {len(records) - n_before}/{len(items)} fiches valides")
    return records


fiche_services = parse_population_fiches()
print(f"\nTotal fiches population : {len(fiche_services)} services")


  ✓ Services_enfants.json: 13/13 fiches valides
  ✓ Services_femmes.json: 12/12 fiches valides
  ✓ Services_personnes_agees.json: 11/11 fiches valides
  ✓ Services_personnes_handicapees.json: 22/22 fiches valides

Total fiches population : 58 services


## 1c. Fusion intelligente (matching flou intra-population)

Les deux sources sont fusionnées **par population cible**, avec un score de
similarité (`rapidfuzz.fuzz.token_sort_ratio`) sur le nom de service
normalisé. Seuil retenu : **80** (cf. `03_Match.ipynb` pour la même logique
appliquée au matching Centre <-> Service).

In [4]:
FUZZY_THRESHOLD = 80


def merge_service_pair(table_rec: dict, fiche_rec: dict) -> dict:
    """table_arabe = libellé officiel; conditions/documents unionés."""
    merged = dict(table_rec)
    merged["service_fr"] = ""  # aucune des deux sources ne fournit de FR (constat honnête)
    merged["description_ar"] = table_rec["description_ar"] or fiche_rec["description_ar"]
    if len(fiche_rec["description_ar"]) > len(merged["description_ar"]):
        merged["description_ar"] = fiche_rec["description_ar"]
    merged["conditions_ar"] = list(dict.fromkeys(table_rec["conditions_ar"] + fiche_rec["conditions_ar"]))
    merged["documents_ar"] = list(dict.fromkeys(table_rec["documents_ar"] + fiche_rec["documents_ar"]))
    merged["institutions"] = list(dict.fromkeys(table_rec["institutions"] + fiche_rec["institutions"]))
    merged["eps_fallback"] = table_rec["eps_fallback"] and fiche_rec["eps_fallback"]
    merged["raw_id"] = fiche_rec.get("raw_id", "")
    merged["sources"] = "table_arabe+fiche_population"
    return merged


def fuse_services(table_services: list, fiche_services: list) -> list:
    by_pop_table = defaultdict(list)
    by_pop_fiche = defaultdict(list)
    for r in table_services:
        by_pop_table[r["population_cible"]].append(r)
    for r in fiche_services:
        by_pop_fiche[r["population_cible"]].append(r)

    fused = []
    match_count = 0
    for population in set(list(by_pop_table) + list(by_pop_fiche)):
        table_group = by_pop_table.get(population, [])
        fiche_group = by_pop_fiche.get(population, [])
        used_fiche_idx = set()

        for t in table_group:
            t_norm = normalize_arabic(t["service_ar"])
            best_idx, best_score = None, 0
            for i, f in enumerate(fiche_group):
                if i in used_fiche_idx:
                    continue
                score = fuzz.token_sort_ratio(t_norm, normalize_arabic(f["service_ar"]))
                if score > best_score:
                    best_idx, best_score = i, score
            if best_idx is not None and best_score >= FUZZY_THRESHOLD:
                fused.append(merge_service_pair(t, fiche_group[best_idx]))
                used_fiche_idx.add(best_idx)
                match_count += 1
            else:
                r = dict(t)
                r["service_fr"] = ""
                r["raw_id"] = ""
                r["sources"] = "table_arabe"
                fused.append(r)

        for i, f in enumerate(fiche_group):
            if i in used_fiche_idx:
                continue
            r = dict(f)
            r["service_fr"] = ""
            r["axe_programme_ar"] = ""
            r["sources"] = "fiche_population"
            fused.append(r)

    print(f"Paires fusionnées (table <-> fiche) : {match_count}")
    return fused


fused_services = fuse_services(table_services, fiche_services)
print(f"Total après fusion : {len(fused_services)} services")


Paires fusionnées (table <-> fiche) : 46
Total après fusion : 71 services


## 1d. Finalisation : ID stable, dédoublonnage global, validation vs manifeste

In [5]:
seen_ids = {}
services_unified = []
for r in fused_services:
    sid = stable_id("svc", "_".join(r["institutions"]), r["service_ar"], r["population_cible"])
    if sid in seen_ids:
        continue  # doublon (déjà vu sous un ID identique -> contenu quasi identique)
    seen_ids[sid] = True
    services_unified.append({
        "id": sid,
        "service_ar": r["service_ar"],
        "service_fr": r.get("service_fr", ""),
        "categorie": r["population_cible"],
        "population_cible_ar": POPULATION_CIBLES.get(r["population_cible"], {}).get("ar", ""),
        "population_cible_fr": POPULATION_CIBLES.get(r["population_cible"], {}).get("fr", ""),
        "axe_programme_ar": r.get("axe_programme_ar", ""),
        "description_ar": r["description_ar"],
        "conditions_ar": r["conditions_ar"],
        "documents_ar": r["documents_ar"],
        "institutions": r["institutions"],
        "eps_fallback": r["eps_fallback"],
        "source": r.get("sources", r.get("source", "")),
        "raw_id": r.get("raw_id", ""),
    })

save_jsonl(PROCESSED_DIR / "services_unified.jsonl", services_unified)

# Validation vs manifeste Services_Sociaux_Organisé.json
manifest = load_json(RAG_DATA_DIR / "Services_Sociaux_Organisé.json")
manifest_total = manifest.get("metadata", {}).get("total_services") if manifest else None
manifest_categories = manifest.get("metadata", {}).get("categories") if manifest else None

with_conditions = sum(1 for s in services_unified if s["conditions_ar"])
with_documents = sum(1 for s in services_unified if s["documents_ar"])
eps_fallback_count = sum(1 for s in services_unified if s["eps_fallback"])

print(f"Services unifiés          : {len(services_unified)}")
print(f"  Manifeste annonce       : {manifest_total} services, catégories {manifest_categories}")
print(f"  Avec conditions         : {with_conditions}/{len(services_unified)} ({100*with_conditions/len(services_unified):.0f}%)")
print(f"  Avec documents          : {with_documents}/{len(services_unified)} ({100*with_documents/len(services_unified):.0f}%)")
print(f"  EPS fallback (aucune institution détectée) : {eps_fallback_count}/{len(services_unified)} ({100*eps_fallback_count/len(services_unified):.0f}%)")
print(f"\nCible guide : >=90% avec conditions+documents -> "
      f"{'OK' if min(with_conditions, with_documents)/len(services_unified) >= 0.9 else 'A SURVEILLER'}")


Services unifiés          : 61
  Manifeste annonce       : 58 services, catégories ['personnes_handicapees', 'personnes_agees', 'enfants', 'femmes']
  Avec conditions         : 61/61 (100%)
  Avec documents          : 61/61 (100%)
  EPS fallback (aucune institution détectée) : 12/61 (20%)

Cible guide : >=90% avec conditions+documents -> OK


## 2. Centres — `centres.jsonl`

Chaque ligne est enveloppée dans une clé `"data"`, avec un champ mal
orthographié `"addresse"` (double s). Fichier signalé bruité (lignes
cassées, région/délégation/commune invalides) — lecture tolérante aux
erreurs via `stream_jsonl_repair`, et validation stricte de la hiérarchie
géographique (jamais de centre orphelin, cf. guide).

In [6]:
def normalize_centre(raw: dict) -> dict:
    d = raw.get("data", raw)  # tolère les deux formats (avec/sans enveloppe "data")
    # Matching flou plutôt qu'exact : la saisie réelle varie beaucoup
    # ("Beni" sans accent, "Oriental" sans "L'", "Hoceima" sans tréma...).
    # Retourne None (rejeté) pour les vraies valeurs aberrantes (champs
    # décalés type nom de délégation/province glissé dans le champ région).
    region = normalize_region(clean_text(d.get("region"))) or ""
    delegation = clean_text(d.get("delegation"))
    commune = clean_text(d.get("commune"))
    nom = clean_text(d.get("nom"))
    adresse = clean_text(d.get("adresse") or d.get("addresse"))
    activite = clean_text(d.get("activite"))
    institutions, eps_fallback = extract_institutions(activite or d.get("description", ""))

    milieu = clean_text(d.get("milieu")).lower()
    if milieu not in ("urbain", "rural", "semi-urbain"):
        milieu = "inconnu"

    def to_float(v):
        try:
            return float(v)
        except (TypeError, ValueError):
            return None

    return {
        "region": region, "delegation": delegation, "commune": commune,
        "nom": nom, "adresse": adresse,
        "milieu": milieu, "propriete": clean_text(d.get("propriete")) or "inconnu",
        "superficie": to_float(d.get("superficie")), "capacite": to_float(d.get("capacite")),
        "activite": activite, "description": clean_text(d.get("description")),
        "programme": clean_text(d.get("programme")), "axe": clean_text(d.get("axe")),
        "categorie2": clean_text(d.get("categorie2")), "personnes_cibles": clean_text(d.get("personnes_cibles")),
        "institutions": institutions, "eps_fallback": eps_fallback,
    }


n_read, n_valid_geo, n_invalid_geo = 0, 0, 0
centres_by_key = {}
for raw in stream_jsonl_repair(RAG_DATA_DIR / "centres.jsonl"):
    n_read += 1
    c = normalize_centre(raw)
    if not (c["region"] in MOROCCAN_REGIONS and c["delegation"] and c["commune"] and c["nom"]):
        n_invalid_geo += 1
        continue
    n_valid_geo += 1
    key = stable_id("centre", c["region"], c["delegation"], c["commune"], c["nom"])
    c["id"] = key
    centres_by_key[key] = c  # dédoublonnage sur (géo complète + nom)

centres_normalized = list(centres_by_key.values())
save_jsonl(PROCESSED_DIR / "centres_normalized.jsonl", centres_normalized)

print(f"Lignes lues (avec réparation)     : {n_read}")
print(f"Géographie valide (12 régions)    : {n_valid_geo} ({100*n_valid_geo/max(n_read,1):.1f}%)")
print(f"Géographie invalide (rejetée)     : {n_invalid_geo}")
print(f"Centres uniques après dédoublonnage: {len(centres_normalized)}")
print(f"\nCible guide : >= 3500 centres valides -> {'OK' if len(centres_normalized) >= 3500 else 'EN DESSOUS'}")


Lignes lues (avec réparation)     : 4987
Géographie valide (12 régions)    : 4504 (90.3%)
Géographie invalide (rejetée)     : 483
Centres uniques après dédoublonnage: 3328

Cible guide : >= 3500 centres valides -> EN DESSOUS


## 3. Programmes 2027 — `DOC-20260716-WA0029.json`

In [7]:
doc = load_json(RAG_DATA_DIR / "DOC-20260716-WA0029.json")
programmes_2027 = []
if doc:
    for p in doc.get("programmes", []):
        pop_cible = p.get("population_cible", [])
        if isinstance(pop_cible, str):
            pop_cible = [pop_cible]
        programmes_2027.append({
            "id": stable_id("prog", p.get("code", ""), p.get("titre_ar", "")),
            "code": p.get("code", ""),
            "titre_ar": clean_text(p.get("titre_ar")),
            "titre_fr": clean_text(p.get("titre_fr")),
            "categorie": p.get("categorie", ""),
            "population_cible": " ".join(pop_cible),
            "description_ar": clean_text(p.get("description")),
            "objectifs_ar": " | ".join(p.get("objectifs", [])),
            "budget": p.get("budget", 0),
            "indicateurs": p.get("indicateurs", []),
            "institutions_liees": p.get("institutions_liees", []),
            "source": "DOC-20260716-WA0029",
        })

save_jsonl(PROCESSED_DIR / "programmes_2027.jsonl", programmes_2027)
budget_total = doc.get("metadata", {}).get("budget_total_mad") if doc else None
print(f"Programmes 2027 : {len(programmes_2027)} (budget total annoncé : {budget_total:,} MAD)"
      if budget_total else f"Programmes 2027 : {len(programmes_2027)}")


Programmes 2027 : 7 (budget total annoncé : 298,000,000 MAD)


## 4. FAQ AOS — `عادل الرحالي.json` (sections imbriquées)

In [8]:
faq_data = load_json(RAG_DATA_DIR / "عادل الرحالي.json")
faq_aos = []
if faq_data:
    for section in faq_data.get("sections", []):
        section_titre = clean_text(section.get("عنوان_القسم"))
        for q in section.get("questions", []):
            question_ar = clean_text(q.get("السؤال"))
            faq_aos.append({
                "id": stable_id("faq", question_ar),
                "question_num": q.get("رقم_السؤال", ""),
                "section_ar": section_titre,
                "question_ar": question_ar,
                "question_fr": "",
                "reponse_ar": clean_text(q.get("الإجابة")),
                "reponse_fr": "",
                "requirements": [], "steps": [], "documents": [],
                "programme": "AOS_2026",
                "source": "عادل الرحالي.json (AOS)",
                "type": "faq",
            })

save_jsonl(PROCESSED_DIR / "faq_aos.jsonl", faq_aos)
print(f"FAQ AOS : {len(faq_aos)} questions")


FAQ AOS : 8 questions


## 5. Rapport de qualité final

In [9]:
report = {
    "services": {
        "total": len(services_unified),
        "avec_conditions": with_conditions,
        "avec_documents": with_documents,
        "eps_fallback_pct": round(100 * eps_fallback_count / max(len(services_unified), 1), 1),
        "manifeste_annonce": manifest_total,
    },
    "centres": {
        "lignes_lues": n_read,
        "valides": n_valid_geo,
        "invalides": n_invalid_geo,
        "taux_bruit_pct": round(100 * n_invalid_geo / max(n_read, 1), 1),
        "uniques": len(centres_normalized),
    },
    "programmes_2027": {"total": len(programmes_2027)},
    "faq_aos": {"total": len(faq_aos)},
}

import json as _json
(REPORTS_DIR / "report_01_normalize.json").write_text(_json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")

print(_json.dumps(report, ensure_ascii=False, indent=2))
print("\n\u2705 01_Normalize termine. Sorties dans data/processed/*.jsonl")


{
  "services": {
    "total": 61,
    "avec_conditions": 61,
    "avec_documents": 61,
    "eps_fallback_pct": 19.7,
    "manifeste_annonce": 58
  },
  "centres": {
    "lignes_lues": 4987,
    "valides": 4504,
    "invalides": 483,
    "taux_bruit_pct": 9.7,
    "uniques": 3328
  },
  "programmes_2027": {
    "total": 7
  },
  "faq_aos": {
    "total": 8
  }
}

✅ 01_Normalize termine. Sorties dans data/processed/*.jsonl
